In [ ]:
import autograd.numpy as anp
import numpy as np
import os
os.chdir('../..')
os.getcwd()
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all = mnist.data.astype(np.float32) / 255.0
y_all = mnist.target.astype(int)

# Subsample train
rng = np.random.default_rng(42)
idx_train = rng.choice(len(X_all), size=5000, replace=False)
remaining = np.setdiff1d(np.arange(len(X_all)), idx_train)
idx_test = rng.choice(remaining, size=2000, replace=False)

X_train, y_train = X_all[idx_train], y_all[idx_train]
X_test, y_test = X_all[idx_test], y_all[idx_test]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")

In [ ]:
from benchmarks_august.targets.bnn import bnn_classification
from benchmarks_august.samplers.warmstart.bnn import adam_fisher
from benchmarks_august.samplers import build_sampler, apply_preprocess

H=20
layers = [X_train.shape[1], H, 10]

# 1. Build target (E and gradE come from here)
target = bnn_classification(X_train, y_train, layer_sizes=layers,
    prior={"kind": "layered_gaussian",
           "sigma_w_layers": [1.0, 1.0],
           "sigma_b_layers": [1.0, 1.0]})

# 2. Warmstart (sets x_ref and Sigma_inv)
ws = adam_fisher(target, n_epochs=3000, lr=3e-3)
target.x_ref = ws["x_ref"]
target.Sigma_inv = ws["Sigma_inv"]

# 3. Build kappa from target.meta
w_prior = 0.5
kappa = np.empty(target.D)
kappa[~target.meta["weight_mask"]] = 1e6
for l, (w_sl, b_sl) in enumerate(target.meta["slices"]):
    sigma_l = target.meta["sigma_w_layers"][l]
    kappa[w_sl] = (1 - w_prior) / w_prior / (sigma_l * np.sqrt(2 * np.pi))

In [ ]:
sampler = build_sampler("boomerang_pli", target, N=10000,
                        refresh_rate=1.0)
apply_preprocess(sampler, target, {"method": "manual"})
sampler.sample_auto()

In [ ]:
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path
t, x = resample_pdmp_path(sampler, n_samples=50000)

In [ ]:
from benchmarks_august.samplers.warmstart.bnn import _forward, _softmax
import numpy as np

burnin = int(len(x) * 0.2)
post = x[burnin:]
idx = np.linspace(0, len(post) - 1, 500, dtype=int)

# Predict on test set
all_probs = np.array([
    _softmax(_forward(post[i], X_test, target.meta["shapes"], target.meta["slices"]))
    for i in idx
])  # (500, n_test, 10)

mean_probs = all_probs.mean(axis=0)  # (n_test, 10)
preds = mean_probs.argmax(axis=1)

acc_test = np.mean(preds == y_test.astype(int))
nll = -np.mean(np.log(np.clip(mean_probs[np.arange(len(y_test)), y_test.astype(int)], 1e-7, 1.0)))

# Adam MAP baseline
logits_adam = _forward(target.x_ref, X_test, target.meta["shapes"], target.meta["slices"])
adam_probs = _softmax(logits_adam)
adam_acc = np.mean(adam_probs.argmax(axis=1) == y_test.astype(int))
adam_nll = -np.mean(np.log(np.clip(adam_probs[np.arange(len(y_test)), y_test.astype(int)], 1e-7, 1.0)))

print(f"{'Method':<25s} {'Acc(test)':>10s} {'NLL':>10s}")
print("─" * 47)
print(f"{'Adam MAP':<25s} {adam_acc:>10.3f} {adam_nll:>10.4f}")
print(f"{'Boomerang PLI':<25s} {acc_test:>10.3f} {nll:>10.4f}")

In [ ]:
print(f"Total simulation time: {sampler.Time[sampler.iteration-1]:.0f}")
print(f"Iterations: {sampler.iteration}")

df = sampler.diagnostics_df
n_bounce = df[df['event_type'] == 'bounce'].shape[0]
n_refresh = df[df['event_type'] == 'refresh'].shape[0]
print(f"Bounces: {n_bounce}, Refreshes: {n_refresh}")
print(f"Refresh ratio: {n_refresh/(n_bounce+n_refresh):.3f}")
print(f"Wall time: {df['wall_seconds'].sum():.1f}s")

# Check bound violations
if 'bound_violations' in df.columns:
    print(f"Bound violations: {df['bound_violations'].sum()}")
    print(f"Max ratio: {df['max_ratio'].max():.2f}")

In [ ]:
import os
import numpy as np

save_dir = "benchmarks_august/results/MNIST"
os.makedirs(save_dir, exist_ok=True)

# Save resampled posterior samples
np.save(f"{save_dir}/boomerang_pli_samples.npy", x)
np.save(f"{save_dir}/boomerang_pli_times.npy", t)

# Save sampler state
np.save(f"{save_dir}/boomerang_pli_positions.npy", sampler.Position[:sampler.iteration])
np.save(f"{save_dir}/boomerang_pli_velocities.npy", sampler.Velocity[:sampler.iteration])
np.save(f"{save_dir}/boomerang_pli_skeleton_times.npy", sampler.Time[:sampler.iteration])

# Save diagnostics
sampler.diagnostics_df.to_csv(f"{save_dir}/boomerang_pli_diagnostics.csv", index=False)

# Save x_ref for reference
np.save(f"{save_dir}/x_ref.npy", target.x_ref)

print(f"Saved to {save_dir}/")